<a href="https://colab.research.google.com/github/heoconngoc/Ruled-Based-A.I.-and-Deep-Learning/blob/main/ALPHA_BETA_TicTacToe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# TITLE: Alpha-Beta Pruning
# =========================================================

"""
Minimax with alpha-beta pruning.
"""

import time
import random

EMPTY=" "
X="X"
O="O"

nodes = 0

def create_board():
    return [EMPTY]*9

def get_moves(b):
    return [i for i in range(9) if b[i]==EMPTY]

def check_winner(b):
    wins=[(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
    for i,j,k in wins:
        if b[i] != EMPTY and b[i] == b[j] == b[k]:
            return b[i]
    return None

def switch(p):
    return O if p==X else X

In [6]:
# ---------------------------
def alphabeta(board, player, alpha, beta):
    global nodes
    nodes += 1

    # Terminal state
    winner=check_winner(board)
    if winner==X: return 1,None
    if winner==O: return -1,None
    if not get_moves(board): return 0,None

    # Max branch
    if player==X:
        value=-999; move=None
        for m in get_moves(board):
            b=board.copy(); b[m]=X
            val,_=alphabeta(b,O,alpha,beta)
            if val>value:
                value=val; move=m
            alpha=max(alpha,value)
            if alpha>=beta:
                break
        return value,move

    # Min branch
    else:
        value=999; move=None
        for m in get_moves(board):
            b=board.copy(); b[m]=O
            val,_=alphabeta(b,X,alpha,beta)
            if val<value:
                value=val; move=m
            beta=min(beta,value)
            if alpha>=beta:
                break
        return value,move


def alpha_beta_agent(board,player):
    _,m=alphabeta(board,player,-999,999)
    return m

In [3]:
# ---------------------------
# GAME ENGINE
# ---------------------------

def play_game(agent1, agent2, verbose=False):
    board = create_board()
    player = X

    while True:

        if player == X:
            move = agent1(board, X)
        else:
            move = agent2(board, O)

        board[move] = player

        if verbose:
            print_board(board)
            print()

        w = check_winner(board)
        if w:
            return 1 if w == X else -1

        if not get_moves(board):
            return 0

        player = switch(player)

In [4]:
# ---------------------------
# EVALUATE WITH NODES + TIME
# ---------------------------
def evaluate(agent1, agent2, games=100, verbose=False):
    global nodes
    results = []
    total_nodes_before = nodes

    start_time = time.time()

    for i in range(games):

        # reset nodes
        nodes_before_game = nodes

        # swap first move for fairness
        if i % 2 == 0:
            res = play_game(agent1, agent2, verbose)
        else:
            res = play_game(agent2, agent1, verbose)
            res = -res

        results.append(res)
        # nodes explored in this game
        nodes_game = nodes - nodes_before_game
        if verbose:
            print(f"Game {i+1}: Nodes explored = {nodes_game}")

    end_time = time.time()
    total_nodes_after = nodes

    print("\n======================")
    print("RESULTS")
    print("======================")

    print("Agent1 Wins :", results.count(1))
    print("Agent2 Wins :", results.count(-1))
    print("Ties        :", results.count(0))

    print("\nTime taken  :", round(end_time - start_time, 4), "sec")
    print("Avg time/game:", round((end_time - start_time)/games, 6), "sec")

    print("\nNodes explored (total):", total_nodes_after - total_nodes_before)
    print("Avg nodes/game:", round((total_nodes_after - total_nodes_before)/games, 2))

In [7]:
# Random agent
def random_agent(board, player):
    return random.choice(get_moves(board))

# Rule-based 1-step agent
def rule_1_step(board, player):

    opponent = switch(player)

    # win
    for m in get_moves(board):
        b = board.copy()
        b[m] = player
        if check_winner(b) == player:
            return m

    # block
    for m in get_moves(board):
        b = board.copy()
        b[m] = opponent
        if check_winner(b) == opponent:
            return m

    return random.choice(get_moves(board))

In [8]:
nodes = 0
evaluate(alpha_beta_agent, random_agent, games=100)


RESULTS
Agent1 Wins : 92
Agent2 Wins : 0
Ties        : 8

Time taken  : 3.2052 sec
Avg time/game: 0.032052 sec

Nodes explored (total): 1133106
Avg nodes/game: 11331.06


In [9]:
nodes = 0
evaluate(alpha_beta_agent, rule_1_step, games=100)


RESULTS
Agent1 Wins : 48
Agent2 Wins : 0
Ties        : 52

Time taken  : 2.1979 sec
Avg time/game: 0.021979 sec

Nodes explored (total): 1128654
Avg nodes/game: 11286.54


In [10]:
nodes = 0
evaluate(alpha_beta_agent, alpha_beta_agent, games=100)


RESULTS
Agent1 Wins : 0
Agent2 Wins : 0
Ties        : 100

Time taken  : 4.1767 sec
Avg time/game: 0.041767 sec

Nodes explored (total): 2165200
Avg nodes/game: 21652.0


Alpha-beta pruning significantly reduces the number of explored nodes while maintaining optimal decisions equivalent to full minimax.